# SmartProcure AI

This notebook mirrors the production pipeline for the **Smart Procurement: Predicting Delivery Delays & Reward-Based Planning Optimization** challenge. It focuses on data inspection, feature engineering, model training, and operational recommendations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import load_source_data, merge_datasets, clean_merged_data, generate_eda_outputs
from src.feature_engineering import create_features, get_model_inputs
from src.evaluate_model import train_and_compare_models, get_feature_importance
from sklearn.model_selection import train_test_split
import pandas as pd

## 1. Load and merge the source datasets

In [ ]:
source_frames = load_source_data(PROJECT_ROOT / "data")
merged_df = merge_datasets(source_frames)
cleaned_df, cleaning_summary = clean_merged_data(merged_df)
cleaned_df.head()

## 2. Generate EDA outputs and inspect the main insights

In [ ]:
eda_insights = generate_eda_outputs(cleaned_df, PROJECT_ROOT / "outputs" / "plots")
pd.Series(eda_insights["narrative_insights"])

## 3. Engineer planning-time features

In [ ]:
feature_df = create_features(cleaned_df)
feature_df.head()

## 4. Train and compare candidate models

In [ ]:
X, y = get_model_inputs(feature_df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
best_result, comparison_df, _ = train_and_compare_models(X_train, X_test, y_train, y_test)
comparison_df

## 5. Inspect the winning model's top features

In [ ]:
feature_importance_df = get_feature_importance(best_result["pipeline"])
feature_importance_df.head(10)

## 6. Review generated recommendation artifacts

In [ ]:
priority_path = PROJECT_ROOT / "outputs" / "delivery_priority_recommendations.csv"
reward_path = PROJECT_ROOT / "outputs" / "reward_simulation_results.csv"

if priority_path.exists():
    display(pd.read_csv(priority_path).head(10))

if reward_path.exists():
    display(pd.read_csv(reward_path))